# T5 — *What if Italy had grown like…?*

**Question (final, locked):** *If you had been working in another European country instead of Italy since the year you started, how much more money would you have earned by today — and what would that amount be worth in concrete things (cars, vacations, rent, Vespas, pizzas)?*

T5 is the personal calculator that closes the project's narrative arc. T1 establishes that Italy's real wages have been flat since 1995 against European peers; T2/T3/T4 break the freeze down geographically, dimensionally and generationally. T5 finally translates the macro fact into a personal number, on a per-user basis: *what has the Italian exception cost YOU specifically?*

**Approach.** OECD Average Annual Wages (constant 2024 USD PPP) for 18 European OECD countries, 1990–2024 — same methodology as T1, just extended to the full European OECD club. We compute year-by-year wage indices (each country indexed to its own start year = 100) and build the calculator: given a user-supplied (start year, current salary, comparator country), it integrates the year-by-year salary differential into a cumulative euro gap, which is then converted into concrete Italian units (cars, vacation weeks, Milano rent, Vespas, pizzas).

**Notebook structure.**

1. **Setup** — imports, palette, paths.
2. **Load OECD AWE** — 18 European OECD countries (raw OECD download covers 26 European members; the panel is filtered to 18 developed-economy comparators in section 2), 1990–2024.
3. **Build wage index panel** — long format, indexed for any (country, base_year) → multiplier vs current.
4. **Italian baseline in EUR** — anchor €30 000 in 2024, back-cast for any earlier start year.
5. **Concrete-unit prices** — 5 fixed 2024 prices (car, vacation week, Milano rent, Vespa, pizza).
6. **Worked example** — *"User started working in Italy in 2000, average salary, vs Germany"* → cumulative gap and unit counts.
7. **Save outputs** — JSON files consumed by the D3.js widget at `site/viz/t5_calculator.html`.


## 1. Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_DIR = Path("data/task_5")
VIZ_DIR  = Path("site/viz")
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Newspaper palette (locked in 02_viz_plan.md)
ACCENT = "#c44536"     # editorial red — used for Italy
INK    = "#1a1a1a"
GRAY   = "#666666"
MID    = "#8a8a8a"
BG     = "#fafafa"
GRID   = "#e0e0e0"


## 2. Load OECD AWE (Europe)

OECD `DSD_EARNINGS@AV_AN_WAGE` — average annual wages, in constant 2024 USD PPP, 1990–2024. The raw download covers 26 European OECD members + the OECD aggregate; the CSV used here has been filtered to 18 developed-economy comparators (see cell below). Methodology consistent with T1 (the Italian series is identical to the millesimo).

In [ ]:
# Note on the dataset: the raw OECD download covers 26 European OECD members
# + the OECD aggregate. The CSV at the path below has been filtered to 18 developed-
# economy comparators (Western/Nordic/Southern Europe) + OECD. The 8 post-Soviet
# OECD members (CZE, EST, HUN, LTU, LVA, POL, SVK, SVN) were dropped at the CSV
# level for editorial coherence — their 1995-2024 wage growth (+89% to +302%)
# reflects post-Soviet convergence, not a comparison comparable with the
# Italian-exception narrative. The original full-coverage download file is
# referenced in SOURCES.md and can be re-fetched from the OECD Data Explorer.

src = DATA_DIR / "wages_real_oecd_europe_1990_2024.csv"
raw = pd.read_csv(src)

# Keep what we need; rename for compactness
wages = (raw[['REF_AREA', 'Reference area', 'TIME_PERIOD', 'OBS_VALUE']]
          .rename(columns={'REF_AREA': 'iso3',
                           'Reference area': 'country',
                           'TIME_PERIOD': 'year',
                           'OBS_VALUE': 'wage_real_usd_ppp_2024'})
          .dropna(subset=['wage_real_usd_ppp_2024'])
          .sort_values(['iso3', 'year'])
          .reset_index(drop=True))

print("Country coverage (after editorial filter):")
print(wages.groupby('iso3').agg(country=('country', 'first'),
                                  first=('year', 'min'),
                                  last=('year', 'max'),
                                  n=('year', 'nunique')).to_string())

# Identify European countries (excluding the OECD aggregate)
EU_OECD = sorted(wages[wages.iso3 != 'OECD'].iso3.unique())
print(f"\n{len(EU_OECD)} European OECD countries: {', '.join(EU_OECD)}")


Country coverage (after editorial filter):
             country  first  last   n
iso3                                 
AUT          Austria   1990  2024  35
BEL          Belgium   1990  2024  35
CHE      Switzerland   1990  2024  35
DEU          Germany   1991  2024  34
DNK          Denmark   1990  2024  35
ESP            Spain   1990  2024  35
FIN          Finland   1990  2024  35
FRA           France   1990  2024  35
GBR   United Kingdom   1990  2024  35
GRC           Greece   1995  2024  30
IRL          Ireland   1990  2024  35
ISL          Iceland   1990  2024  35
ITA            Italy   1990  2024  35
LUX       Luxembourg   1990  2024  35
NLD      Netherlands   1990  2024  35
NOR           Norway   1990  2024  35
OECD            OECD   1990  2024  35
PRT         Portugal   1995  2024  30
SWE           Sweden   1990  2024  35

18 European OECD countries: AUT, BEL, CHE, DEU, DNK, ESP, FIN, FRA, GBR, GRC, IRL, ISL, ITA, LUX, NLD, NOR, PRT, SWE


## 3. Build wage index panel

For the calculator we need, for every (country, year) pair, an *index* expressed in `1995 = 100`. The user's per-year-of-career growth ratio is then `idx[country][year_T] / idx[country][year_start]`, regardless of euro level.

Why 1995 as the unified base year: it is the latest single year in which **every** country in our 18-country panel has data. Earlier years have gaps (ex-Soviet countries start in 1995, Germany has a 1990 reunification break). Anchoring at 1995 gives us a clean common reference.

In [ ]:
BASE_YEAR = 1995

# Index each country's wage to its own 1995 value = 100
def index_to_1995(group):
    base = group.loc[group.year == BASE_YEAR, 'wage_real_usd_ppp_2024']
    if base.empty:
        # Country lacks 1995 data; index to its first available year and flag
        first = group.year.min()
        base = group.loc[group.year == first, 'wage_real_usd_ppp_2024']
        group['index_first_year'] = first
    else:
        group['index_first_year'] = BASE_YEAR
    base_val = base.iloc[0]
    group['idx_1995'] = group['wage_real_usd_ppp_2024'] / base_val * 100
    return group

panel = (wages.groupby('iso3', group_keys=False)
              .apply(index_to_1995)
              .reset_index(drop=True))

# Show the headline finding: Δ% 1995 → 2024
gap_table = (panel[panel.year.isin([1995, 2024]) & (panel.iso3 != 'OECD')]
             .pivot(index='iso3', columns='year', values='idx_1995')
             .dropna()
             .assign(growth_pct=lambda d: d[2024] - 100)
             .sort_values('growth_pct', ascending=False))

print("Real-wage growth 1995 → 2024 (constant USD PPP, OECD AWE):")
print(gap_table[[2024, 'growth_pct']].round(1).to_string())


Real-wage growth 1995 → 2024 (constant USD PPP, OECD AWE):
year   2024  growth_pct
iso3                   
ISL   171.1        71.1
NOR   164.5        64.5
SWE   162.1        62.1
IRL   160.2        60.2
GBR   140.3        40.3
LUX   139.8        39.8
DNK   136.9        36.9
FIN   129.5        29.5
FRA   126.6        26.6
CHE   125.9        25.9
GRC   122.5        22.5
DEU   121.5        21.5
PRT   121.2        21.2
AUT   119.6        19.6
BEL   117.4        17.4
NLD   105.6         5.6
ESP   104.8         4.8
ITA   102.7         2.7


/sessions/festive-great-ritchie/tmp/ipykernel_4/2296881894.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(index_to_1995)


## 4. Italian baseline in EUR (default for the calculator)

The user-facing widget lives in euros, not USD PPP. We anchor a default Italian baseline in 2024 (€30 000 — full-time gross annual wage, ISTAT-aligned, see `SOURCES.md` § Dataset T5.2) and back-cast it to earlier years using Italy's own OECD wage growth. The user can override the baseline at any time with a custom value.

In [ ]:
BASELINE_EUR_2024 = 30_000  # Italian default (ISTAT full-time gross annual, rounded)

italy = panel[panel.iso3 == 'ITA'].set_index('year')['idx_1995']
italy_2024_idx = italy.loc[2024]

italy_baseline_eur = pd.DataFrame({
    'year': italy.index,
    'idx_1995': italy.values,
    'baseline_eur': BASELINE_EUR_2024 * (italy.values / italy_2024_idx),
}).sort_values('year').reset_index(drop=True)

print("Implied Italian baseline gross annual wage in EUR, by start year:")
print(italy_baseline_eur.round({'idx_1995': 2, 'baseline_eur': 0}).to_string(index=False))


Implied Italian baseline gross annual wage in EUR, by start year:
 year  idx_1995  baseline_eur
 1990    104.42       30496.0
 1991    105.30       30752.0
 1992    105.20       30722.0
 1993    103.58       30249.0
 1994    102.24       29858.0
 1995    100.00       29204.0
 1996    100.79       29435.0
 1997    103.68       30280.0
 1998    104.01       30376.0
 1999    105.20       30722.0
 2000    105.18       30716.0
 2001    105.92       30932.0
 2002    105.17       30713.0
 2003    104.89       30631.0
 2004    106.88       31214.0
 2005    108.34       31638.0
 2006    109.02       31838.0
 2007    108.96       31822.0
 2008    108.82       31779.0
 2009    109.65       32023.0
 2010    110.59       32296.0
 2011    108.82       31781.0
 2012    105.37       30772.0
 2013    105.72       30873.0
 2014    106.13       30993.0
 2015    107.05       31261.0
 2016    107.90       31511.0
 2017    107.21       31310.0
 2018    107.33       31345.0
 2019    107.87       31501.0
 202

## 5. Concrete-unit prices (EUR 2024, fixed)

Five Italian goods, priced at 2024 euros. The widget converts the cumulative wage gap into "how many of these you could have bought". Prices are sourced in `SOURCES.md` § Dataset T5.3.

In [ ]:
CONCRETE_UNITS = [
    {
        'key':         'fiat_panda',
        'label':       'new Fiat Panda',
        'label_plural':'new Fiat Pandas',
        'price_eur':   18_000,
        'icon':        '🚗',
        'note':        'Entry-level new compact car, ANFIA listino base 2024',
    },
    {
        'key':         'vacation_week',
        'label':       'week of vacation on the Italian coast',
        'label_plural':'weeks of vacation on the Italian coast',
        'price_eur':   1_200,
        'icon':        '🏖️',
        'note':        '1-week beach package for two, Federalberghi/Confturismo average 2024',
    },
    {
        'key':         'milano_rent',
        'label':       'month of rent in a Milan studio apartment',
        'label_plural':'months of rent in a Milan studio apartment',
        'price_eur':   900,
        'icon':        '🏠',
        'note':        'Studio apartment ≤ 35 sqm in Milan, Immobiliare.it median 2024',
    },
    {
        'key':         'vespa',
        'label':       'Vespa Primavera 125',
        'label_plural':'Vespa Primavera 125s',
        'price_eur':   4_800,
        'icon':        '🛵',
        'note':        'Piaggio listino 2024',
    },
    {
        'key':         'pizza',
        'label':       'restaurant margherita pizza',
        'label_plural':'restaurant margherita pizzas',
        'price_eur':   8,
        'icon':        '🍕',
        'note':        'National average, FIPE 2024',
    },
]

print("Concrete units (all in EUR 2024, fixed prices):\n")
for u in CONCRETE_UNITS:
    print(f"  {u['icon']}  {u['label_plural']:50s} €{u['price_eur']:>6,}")


Concrete units (all in EUR 2024, fixed prices):

  🚗  new Fiat Pandas                                    €18,000
  🏖️  weeks of vacation on the Italian coast             € 1,200
  🏠  months of rent in a Milan studio apartment         €   900
  🛵  Vespa Primavera 125s                               € 4,800
  🍕  restaurant margherita pizzas                       €     8


## 6. Worked example — *"I started working in Italy in 2000; what if I had been in Germany?"*

Compute the cumulative real-wage gap from the start year to 2024, applying:

- Italy's growth trajectory to the user's actual salary (the "real" career)
- Germany's growth trajectory to the same starting salary (the "what-if" career)
- Difference summed year by year = cumulative gap

In [ ]:
def compute_career_gap(start_year, comparator_iso3,
                       baseline_year=None, baseline_eur=None):
    """
    Compute the year-by-year and cumulative real-wage gap between Italy and a
    comparator country, anchored on the user's chosen baseline.

    If baseline_year/baseline_eur are not given, the function uses the
    implied Italian baseline (€30k in 2024 back-cast to the start_year).
    """
    italy_idx = panel[panel.iso3 == 'ITA'].set_index('year')['idx_1995']
    comp_idx  = panel[panel.iso3 == comparator_iso3].set_index('year')['idx_1995']

    # Effective baseline year: respects countries with shorter coverage
    comp_first = panel[panel.iso3 == comparator_iso3].year.min()
    effective_start = max(start_year, comp_first)

    # Default baseline: Italian implied wage in the effective_start year
    if baseline_eur is None:
        baseline_eur = BASELINE_EUR_2024 * (italy_idx.loc[effective_start] / italy_idx.loc[2024])
        baseline_year = effective_start

    # Year-by-year salary in each scenario
    years = sorted(set(italy_idx.index) & set(comp_idx.index))
    years = [y for y in years if y >= effective_start]

    rows = []
    for y in years:
        salary_it = baseline_eur * (italy_idx.loc[y] / italy_idx.loc[effective_start])
        salary_cp = baseline_eur * (comp_idx.loc[y]  / comp_idx.loc[effective_start])
        rows.append({'year': y,
                     'salary_it': salary_it,
                     'salary_cp': salary_cp,
                     'gap_year': salary_cp - salary_it})

    df = pd.DataFrame(rows)
    df['gap_cum'] = df['gap_year'].cumsum()

    return {
        'effective_start': effective_start,
        'baseline_eur': baseline_eur,
        'comparator': comparator_iso3,
        'final_year': df.year.max(),
        'final_gap_year': df['gap_year'].iloc[-1],
        'final_gap_cum': df['gap_cum'].iloc[-1],
        'trajectory': df,
    }

# Example
ex = compute_career_gap(start_year=2000, comparator_iso3='DEU')
print(f"User scenario: started working in Italy in {ex['effective_start']}, default baseline €{ex['baseline_eur']:,.0f}.")
print(f"Comparator: Germany.")
print(f"\n  Final year ({ex['final_year']}) salary:")
print(f"    Italian trajectory: €{ex['trajectory']['salary_it'].iloc[-1]:>10,.0f}")
print(f"    German  trajectory: €{ex['trajectory']['salary_cp'].iloc[-1]:>10,.0f}")
print(f"    Single-year gap:    €{ex['final_gap_year']:>10,.0f}")
print(f"\n  CUMULATIVE GAP {ex['effective_start']}–{ex['final_year']}: €{ex['final_gap_cum']:,.0f}")
print(f"\nIn concrete Italian units, that's:")
for u in CONCRETE_UNITS:
    n = ex['final_gap_cum'] / u['price_eur']
    print(f"  {u['icon']}  {n:>8,.1f}  {u['label_plural']}")


User scenario: started working in Italy in 2000, default baseline €30,716.
Comparator: Germany.

  Final year (2024) salary:
    Italian trajectory: €    30,000
    German  trajectory: €    35,932
    Single-year gap:    €     5,932

  CUMULATIVE GAP 2000–2024: €51,907

In concrete Italian units, that's:
  🚗       2.9  new Fiat Pandas
  🏖️      43.3  weeks of vacation on the Italian coast
  🏠      57.7  months of rent in a Milan studio apartment
  🛵      10.8  Vespa Primavera 125s
  🍕   6,488.4  restaurant margherita pizzas


## 7. Save outputs (JSON for the D3 widget)

The interactive widget at `site/viz/t5_calculator.html` consumes a single bundled file containing three sub-objects:

- **`wage_indices`** — country × year → indexed wage (1995 = 100, with each country's first available year flagged).
- **`italian_baseline_eur`** — year → implied Italian baseline EUR (€30 000 in 2024, back-cast).
- **`concrete_units`** — the 5 unit prices and labels.

The bundle is saved next to the widget in two forms:

- `site/viz/t5_calculator_data.json` — fetched in a single HTTP request when the page is served over http(s).
- `site/viz/t5_calculator_data.js` — the same bundle wrapped in a JS variable assignment, used as a fallback when the page is opened via `file://` (double-click), where `fetch()` is blocked.

In [ ]:
# Build wage_indices.json
country_meta = (panel[panel.iso3 != 'OECD']
                .groupby('iso3')
                .agg(country=('country', 'first'),
                     first_year=('year', 'min'),
                     last_year=('year', 'max'))
                .reset_index())

wage_indices = {}
for _, row in country_meta.iterrows():
    iso = row['iso3']
    series = (panel[panel.iso3 == iso][['year', 'idx_1995']]
              .sort_values('year'))
    wage_indices[iso] = {
        'country':        row['country'],
        'first_year':     int(row['first_year']),
        'last_year':      int(row['last_year']),
        'index_base':     BASE_YEAR,
        'series':         {int(y): round(v, 4) for y, v in zip(series.year, series.idx_1995)},
    }

# We don't persist the three sub-objects to disk separately; the widget only
# consumes the bundled file built below. The in-memory `wage_indices` dict is
# already populated above; build the matching `italian_baseline` dict next so
# both are ready for the bundle.
italian_baseline = {
    'baseline_2024_eur': BASELINE_EUR_2024,
    'baseline_source':   'ISTAT full-time gross annual wage, rounded to €30 000 in 2024',
    'series': {int(y): round(v, 0) for y, v in
               zip(italy_baseline_eur.year, italy_baseline_eur.baseline_eur)},
}
print(f"Built wage_indices: {len(wage_indices)} countries")
print(f"Built italian_baseline: {len(italian_baseline['series'])} years")
print(f"Built concrete_units: {len(CONCRETE_UNITS)} units")

# Bundled JSON
bundle = {
    'wage_indices':         wage_indices,
    'italian_baseline_eur': italian_baseline,
    'concrete_units':       CONCRETE_UNITS,
    'meta': {
        'index_base_year':       BASE_YEAR,
        'baseline_anchor_year':  2024,
        'baseline_anchor_eur':   BASELINE_EUR_2024,
        'data_source':           'OECD AWE constant 2024 USD PPP + ISTAT default + Italian retail prices 2024',
        'editorial_filter':
            "The raw OECD download covers 26 European OECD members; this bundle is the editorial subset of "
            "18 developed-economy comparators (Western, Nordic, Southern Europe). The 8 post-Soviet members "
            "(CZE, EST, HUN, LTU, LVA, POL, SVK, SVN) were dropped at the CSV-load step because their "
            "1995-2024 wage growth (+89% to +302%) reflects post-Soviet convergence rather than a productivity "
            "dynamic comparable with the Italian-exception narrative. See data/task_5/SOURCES.md for the full justification.",
    },
}
(VIZ_DIR / 't5_calculator_data.json').write_text(json.dumps(bundle, indent=1, ensure_ascii=False))
print(f"Saved bundle JSON: {VIZ_DIR / 't5_calculator_data.json'}")

# JS-wrapped form
js_payload = (
    "// Auto-generated by Task_5.ipynb. Inlines the T5 calculator data so the\n"
    "// widget works both via file:// (double-click) and via http(s) servers.\n"
    "window.T5_DATA = " + json.dumps(bundle, ensure_ascii=False) + ";\n"
)
(VIZ_DIR / 't5_calculator_data.js').write_text(js_payload)
print(f"Saved bundle JS:   {VIZ_DIR / 't5_calculator_data.js'}")


Saved: data/task_5/wage_indices.json — 18 countries
Saved: data/task_5/italian_baseline_eur.json — 35 years
Saved: data/task_5/concrete_units.json — 5 units
Saved bundle JSON: site/viz/t5_calculator_data.json
Saved bundle JS:   site/viz/t5_calculator_data.js


## Outputs

- `site/viz/t5_calculator_data.json` — bundled single-fetch file for the D3 widget (http(s) path).
- `site/viz/t5_calculator_data.js` — same bundle wrapped as a JS variable, used as a fallback for the `file://` path.

Documentation: `data/task_5/SOURCES.md`.

The JS calculator widget that consumes these files is at `site/viz/t5_calculator.html`, embedded as Figure 6 in `site/index.html`.
